# Compute mean change confidence intervals

Full temporal coverage: compute NDVI change confidence intervals for all same-month consecutive-year pairs from 2019 to 2025 across all study areas. Results and configuration are saved under `outputs/change_ci/all_year_pairs_2019_2025`.

In [6]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'change_ci').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from change_ci import AnalysisConfig, ChangeCIAnalysis, build_pairs, discover_rasters

## Phase 1 configuration

In [7]:
DATA_ROOT = ROOT / 'outputs' / 'composites'
OUTPUT_DIR = ROOT / 'outputs' / 'change_ci' / 'all_year_pairs_2019_2025'

config = AnalysisConfig(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    study_areas=None,
    months=('01', '05', '09', '11'),
    year_range=(2019, 2025),
    index_band='NDVI',
    engine='fft',
    tau=0.05,
    tau_sensitivity=(0.02, 0.05, 0.10),
    alpha=0.05,
    max_lag_fraction=0.25,
    min_n_eff=30,
)
config.to_dict()

{'data_root': '/home/tshihadah/code/wadi-gaza-research/outputs/composites',
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/all_year_pairs_2019_2025',
 'study_areas': None,
 'months': ('01', '05', '09', '11'),
 'year_range': (2019, 2025),
 'index_band': 'NDVI',
 'engine': 'fft',
 'tau': 0.05,
 'tau_sensitivity': (0.02, 0.05, 0.1),
 'alpha': 0.05,
 'max_lag_fraction': 0.25,
 'min_n_eff': 30}

## Discover all same-month consecutive-year pairs

Expected coverage: 168 pairs (seven study areas × four months × six year transitions).

In [8]:
raster_records = discover_rasters(config.data_root)
pair_specs = build_pairs(raster_records, config)
pair_table = pd.DataFrame([
    {
        'study_area': pair.study_area,
        'month': pair.month,
        'year_start': pair.year_start,
        'year_end': pair.year_end,
        'earlier_path': pair.earlier_path,
        'later_path': pair.later_path,
    }
    for pair in pair_specs
])
pair_table

,study_area,month,year_start,year_end,earlier_path,later_path
0,AlAhrashProtectorate,01,2019,2020,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
1,AlAhrashProtectorate,01,2020,2021,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
2,AlAhrashProtectorate,01,2021,2022,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
3,AlAhrashProtectorate,01,2022,2023,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
4,AlAhrashProtectorate,01,2023,2024,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
...,...,...,...,...,...,...
163,WadiGaza,11,2020,2021,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
164,WadiGaza,11,2021,2022,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
165,WadiGaza,11,2022,2023,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
166,WadiGaza,11,2023,2024,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...


## Run and save

In [9]:
analysis = ChangeCIAnalysis(config)
results, correlograms = analysis.run(pair_specs)
analysis.save(results, correlograms)
print(f'Saved {len(results)} result rows and {len(correlograms)} correlogram rows to {OUTPUT_DIR}')

Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.999368875690768, 9.989040393093362)
Raster pixels are not square: (9.9

Saved 504 result rows and 12744 correlogram rows to /home/tshihadah/code/wadi-gaza-research/outputs/change_ci/all_year_pairs_2019_2025


## Results and review flags

In [10]:
primary = results.loc[results['is_primary_tau']].copy()
summary_columns = [
    'study_area', 'month', 'year_start', 'year_end', 'n', 'mean_change', 'sd',
    'range_px', 'range_map_units', 'n_eff', 'ci_low', 'ci_high',
    'naive_ci_low', 'naive_ci_high', 'flag_no_range', 'flag_low_n_eff', 'error',
]
display(primary[summary_columns].sort_values('study_area'))

review = primary.loc[
    primary['flag_no_range']
    | primary['flag_low_n_eff']
    | primary['error'].fillna('').ne('')
]
print(f'{len(review)} primary-threshold result(s) flagged or failed.')
display(review[summary_columns].sort_values('study_area'))

,study_area,month,year_start,year_end,n,mean_change,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
1,AlAhrashProtectorate,01,2019,2020,34251,0.025459,0.047953,26.0,259.983591,97.242890,0.015807,0.035111,0.024951,0.025967,False,False,
70,AlAhrashProtectorate,11,2024,2025,34251,0.021191,0.024146,NaN,NaN,NaN,NaN,NaN,0.020935,0.021446,True,False,
67,AlAhrashProtectorate,11,2023,2024,34251,-0.038997,0.053865,NaN,NaN,NaN,NaN,NaN,-0.039567,-0.038426,True,False,
64,AlAhrashProtectorate,11,2022,2023,34251,0.024167,0.040288,44.0,439.972231,38.581695,0.011032,0.037302,0.023740,0.024594,False,False,
61,AlAhrashProtectorate,11,2021,2022,34251,0.013240,0.027787,45.0,449.971599,47.982587,0.005170,0.021310,0.012946,0.013534,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,WadiGaza,01,2020,2021,29452,0.078356,0.184598,15.0,150.001958,245.397747,0.055145,0.101567,0.076248,0.080464,False,False,
433,WadiGaza,01,2019,2020,29455,-0.060556,0.179750,11.0,110.001436,371.073834,-0.078905,-0.042207,-0.062608,-0.058503,False,False,
499,WadiGaza,11,2023,2024,29455,-0.155242,0.191373,33.0,330.004308,80.216575,-0.197770,-0.112713,-0.157427,-0.153056,False,False,
463,WadiGaza,05,2023,2024,29455,-0.100154,0.144625,28.0,280.003655,123.888102,-0.125875,-0.074434,-0.101806,-0.098503,False,False,


25 primary-threshold result(s) flagged or failed.


,study_area,month,year_start,year_end,n,mean_change,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
25,AlAhrashProtectorate,05,2021,2022,34251,0.018505,0.026009,NaN,NaN,NaN,NaN,NaN,0.018229,0.018780,True,False,
31,AlAhrashProtectorate,05,2023,2024,34251,-0.027818,0.044327,NaN,NaN,NaN,NaN,NaN,-0.028287,-0.027348,True,False,
34,AlAhrashProtectorate,05,2024,2025,34251,0.006937,0.020485,NaN,NaN,NaN,NaN,NaN,0.006720,0.007154,True,False,
49,AlAhrashProtectorate,09,2023,2024,34251,-0.017210,0.037026,NaN,NaN,NaN,NaN,NaN,-0.017602,-0.016817,True,False,
52,AlAhrashProtectorate,09,2024,2025,34251,0.005412,0.019754,NaN,NaN,NaN,NaN,NaN,0.005202,0.005621,True,False,
67,AlAhrashProtectorate,11,2023,2024,34251,-0.038997,0.053865,NaN,NaN,NaN,NaN,NaN,-0.039567,-0.038426,True,False,
70,AlAhrashProtectorate,11,2024,2025,34251,0.021191,0.024146,NaN,NaN,NaN,NaN,NaN,0.020935,0.021446,True,False,
79,BesorStream,01,2021,2022,11725,-0.058771,0.107155,NaN,NaN,NaN,NaN,NaN,-0.060711,-0.056831,True,False,
82,BesorStream,01,2022,2023,11725,-0.036994,0.101245,NaN,NaN,NaN,NaN,NaN,-0.038827,-0.035161,True,False,
88,BesorStream,01,2024,2025,11725,-0.265244,0.134374,NaN,NaN,NaN,NaN,NaN,-0.267676,-0.262811,True,False,
